# LightGBMRanker: Learning-to-Rank with LambdaRank

This notebook demonstrates how to build a learning-to-rank pipeline using `LightGBMRanker` with the `lambdarank` objective, which optimises NDCG directly.

```python
from recommenders.models.lightgbm import LightGBMRanker

ranker = LightGBMRanker(params={"objective": "lambdarank", "metric": "ndcg", ...})
ranker.fit(train_x, train_y, valid_x, valid_y,
           train_group=train_group, valid_group=valid_group)
scores  = ranker.predict(test_x)
ranker.save("model.lgb")

loaded  = LightGBMRanker.load("model.lgb")
scores  = loaded.predict(test_x)
```

## 0. Imports and Parameters

In [ ]:
import os
import sys
import numpy as np
from tempfile import TemporaryDirectory
from sklearn.metrics import ndcg_score as sklearn_ndcg

import recommenders.datasets.criteo as criteo
from recommenders.models.lightgbm import LightGBMRanker
from recommenders.models.lightgbm.lightgbm_utils import NumEncoder
from recommenders.utils.notebook_utils import store_metadata

print(f"Python: {sys.version}")

In [ ]:
# Data settings
SIZE      = "sample"  # 'sample' | 'full'
LABEL_COL = "Label"
NUME_COLS = [f"I{i}" for i in range(1, 14)]
CATE_COLS = [f"C{i}" for i in range(1, 27)]
HEADER    = [LABEL_COL] + NUME_COLS + CATE_COLS

# Query group settings (simulated, since Criteo has no user IDs)
MIN_GROUP_SIZE = 5
MAX_GROUP_SIZE = 20

# LightGBMRanker parameters (lambdarank)
PARAMS = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [5, 10],
    "num_leaves": 64,
    "learning_rate": 0.05,
    "feature_fraction": 0.8,
    "num_threads": 4,
}
NUM_BOOST_ROUND       = 200
EARLY_STOPPING_ROUNDS = 20

## 1. Data Preparation

We use the [Criteo dataset](https://www.kaggle.com/c/criteo-display-ad-challenge) as a source of binary relevance labels (click / no-click) and split it into train (80%), validation (10%), and test (10%) sets.

Because Criteo has no user or session IDs, we **simulate query groups** by partitioning each split into contiguous chunks of random size (`MIN_GROUP_SIZE`–`MAX_GROUP_SIZE`). Each chunk represents a ranked list of candidate items for one query.

In [ ]:
with TemporaryDirectory() as tmp:
    all_data = criteo.load_pandas_df(size=SIZE, local_cache_path=tmp, header=HEADER)

length     = len(all_data)
train_data = all_data.iloc[: int(0.8 * length)]
valid_data = all_data.iloc[int(0.8 * length) : int(0.9 * length)]
test_data  = all_data.iloc[int(0.9 * length) :]

def make_groups(n, seed):
    """Partition n items into contiguous query groups of random size."""
    rng = np.random.default_rng(seed)
    groups = []
    total = 0
    while total < n:
        g = int(rng.integers(MIN_GROUP_SIZE, MAX_GROUP_SIZE + 1))
        groups.append(min(g, n - total))
        total += groups[-1]
    return np.array(groups)

train_group = make_groups(len(train_data), seed=42)
valid_group = make_groups(len(valid_data), seed=43)
test_group  = make_groups(len(test_data),  seed=44)

print(f"train: {len(train_data):,}  valid: {len(valid_data):,}  test: {len(test_data):,}")
print(f"train queries: {len(train_group):,}  valid: {len(valid_group):,}  test: {len(test_group):,}")

## 2. Feature Encoding (NumEncoder)

`NumEncoder` converts all categorical columns into dense numerical features through three steps:
1. **Low-frequency filtering** — rare categories become `<LESS>`, missing values become `<UNK>`.
2. **Sequential target & count encoding** — encodes each sample using statistics from previous samples only (no label leakage).
3. **Binary encoding** — the ordinal-encoded category values are expanded into bit vectors.

In [ ]:
encoder = NumEncoder(CATE_COLS, NUME_COLS, LABEL_COL)

train_x, train_y = encoder.fit_transform(train_data)
valid_x, valid_y = encoder.transform(valid_data)
test_x,  test_y  = encoder.transform(test_data)

print(f"train_x: {train_x.shape}  valid_x: {valid_x.shape}  test_x: {test_x.shape}")

## 3. Training (LightGBMRanker.fit)

Pass `train_group` and `valid_group` so LightGBM knows which items belong to the same query. Early stopping monitors NDCG on the validation set.

In [ ]:
ranker = LightGBMRanker(
    params=PARAMS,
    num_boost_round=NUM_BOOST_ROUND,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
)

ranker.fit(
    train_x, train_y,
    valid_x=valid_x, valid_y=valid_y,
    train_group=train_group, valid_group=valid_group,
)

## 4. Inference and Evaluation (LightGBMRanker.predict)

Score all test items and compute **NDCG@5** and **NDCG@10** averaged across queries. Queries where all labels are 0 (no relevant item) are excluded from the average.

In [ ]:
scores = ranker.predict(test_x)

ndcg5_list, ndcg10_list = [], []
start = 0
for g in test_group:
    end = start + g
    y_true  = test_y[start:end].reshape(1, -1)
    y_score = scores[start:end].reshape(1, -1)
    if y_true.sum() > 0:  # skip queries with no relevant item
        ndcg5_list.append(sklearn_ndcg(y_true, y_score, k=5))
        ndcg10_list.append(sklearn_ndcg(y_true, y_score, k=10))
    start = end

ndcg_at_5  = float(np.mean(ndcg5_list))
ndcg_at_10 = float(np.mean(ndcg10_list))

print(f"NDCG@5:  {ndcg_at_5:.4f}")
print(f"NDCG@10: {ndcg_at_10:.4f}")

In [ ]:
# Record results for tests - ignore this cell
store_metadata("ndcg_at_5",  ndcg_at_5)
store_metadata("ndcg_at_10", ndcg_at_10)

## 5. Top-K Ranking Example

In a real recommendation pipeline, scores are computed per query (user session) and items are sorted by descending score. Below we display the top-5 items for the first few test queries.

In [ ]:
import pandas as pd

start = 0
for query_id, g in enumerate(test_group[:3]):  # show first 3 queries
    end = start + g
    query_df = pd.DataFrame({
        "score": scores[start:end],
        "label": test_y[start:end].reshape(-1),
    })
    top5 = query_df.nlargest(5, "score").reset_index(drop=True)
    print(f"Query {query_id} (size={g})  top-5:")
    display(top5)
    start = end

## 6. Save and Load (save / load)

A model saved with `save()` can be restored with `LightGBMRanker.load()` and produces identical predictions.

In [ ]:
with TemporaryDirectory() as tmp:
    model_path = os.path.join(tmp, "ranker.lgb")

    ranker.save(model_path)

    loaded_ranker = LightGBMRanker.load(model_path)
    loaded_scores = loaded_ranker.predict(test_x)

assert np.allclose(scores, loaded_scores), "Predictions differ after save/load!"
print("Save/load verified: predictions are identical.")